In [ ]:
import sys

sys.path.append("../..")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.cm import inferno
from matplotlib.colors import Normalize
from numba import njit
from skimage.color import rgb2lab
from skimage.data import astronaut
from skimage.segmentation import mark_boundaries, slic

In [ ]:
from deephierarchy.core import C4
from deephierarchy.graph import build_rag, weight_rag_weighted_area_dist
from deephierarchy.labeling import compute_area, compute_sum, compute_sum_p

In [ ]:
img = astronaut()

In [ ]:
label_map = slic(img, start_label=0)

In [ ]:
rag = build_rag(label_map, C4)

In [ ]:
area = compute_area(label_map)
sum_v = compute_sum(label_map, rgb2lab(img))
sum_p = compute_sum_p(label_map)
centroid = sum_p / area[:, None]

In [ ]:
rag_weights = weight_rag_weighted_area_dist(rag, area, sum_v)

In [ ]:
weight_set = np.unique(rag_weights[rag.adjacency_matrix])
min_w = weight_set.min()
max_w = weight_set.max()
norm = Normalize(vmin=min_w, vmax=max_w)

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(mark_boundaries(img, label_map))
for n1 in range(rag.num_nodes):
    for n2 in range(n1, rag.num_nodes):
        if rag.adjacency_matrix[n1, n2]:
            plt.plot(
                (centroid[n1, 1], centroid[n2, 1]),
                (centroid[n1, 0], centroid[n2, 0]),
                c=inferno(norm(rag_weights[n1, n2])),
            )
plt.scatter(centroid[:, 1], centroid[:, 0], c="r")